# CC12M Image Downloader (Resume-Safe)

This notebook downloads all images listed in `dataset/cc12m.tsv` into `dataset/cc12m_image_cache`.

Behavior:
- Skips images that are already cached
- Retries failed downloads
- Logs failures to `dataset/cc12m_image_cache/_failed_downloads.tsv`
- Safe to rerun multiple times

In [ ]:
import os
import io
import time
import hashlib
from urllib.parse import urlparse
from urllib.request import Request, urlopen
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

from PIL import Image
from tqdm.auto import tqdm

TSV_PATH = 'dataset/cc12m.tsv'
CACHE_DIR = 'dataset/cc12m_image_cache'
NUM_WORKERS = min(64, (os.cpu_count() or 8) * 4)
MAX_PENDING = NUM_WORKERS * 4
TIMEOUT_SECONDS = 30
RETRIES = 3
VERIFY_EXISTING = False

print('TSV_PATH:', TSV_PATH)
print('CACHE_DIR:', CACHE_DIR)
print('NUM_WORKERS:', NUM_WORKERS)
print('MAX_PENDING:', MAX_PENDING)
print('TIMEOUT_SECONDS:', TIMEOUT_SECONDS)
print('RETRIES:', RETRIES)
print('VERIFY_EXISTING:', VERIFY_EXISTING)

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}


def cache_path_from_url(image_url, cache_dir):
    parsed = urlparse(str(image_url))
    ext = os.path.splitext(parsed.path)[1].lower()
    if ext not in VALID_EXTS:
        ext = '.img'
    key = hashlib.sha1(str(image_url).encode('utf-8')).hexdigest()
    return os.path.join(cache_dir, key + ext)


def is_cached(path, verify=False):
    if not os.path.isfile(path):
        return False
    if not bool(verify):
        return True
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        try:
            os.remove(path)
        except Exception:
            pass
        return False


def download_to_cache(image_url, cache_path, timeout_seconds=30, retries=3):
    if os.path.isfile(cache_path):
        return ('skipped', '')

    req = Request(str(image_url), headers={'User-Agent': 'Mozilla/5.0'})
    last_error = ''

    for attempt in range(max(1, int(retries))):
        tmp_path = cache_path + '.tmp'
        try:
            with urlopen(req, timeout=float(timeout_seconds)) as resp:
                data = resp.read()

            if not data:
                raise RuntimeError('empty response body')

            with Image.open(io.BytesIO(data)) as img:
                img.verify()

            with open(tmp_path, 'wb') as f:
                f.write(data)
            os.replace(tmp_path, cache_path)
            return ('downloaded', '')
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            try:
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)
            except Exception:
                pass
            if attempt + 1 < max(1, int(retries)):
                time.sleep(min(2 ** attempt, 5))

    return ('failed', last_error)


def run_cc12m_cache_download(
    tsv_path,
    cache_dir,
    num_workers=32,
    max_pending=128,
    timeout_seconds=30,
    retries=3,
    verify_existing=False,
):
    if not os.path.isfile(tsv_path):
        raise FileNotFoundError(f'CC12M TSV not found: {tsv_path}')

    os.makedirs(cache_dir, exist_ok=True)
    failed_log = os.path.join(cache_dir, '_failed_downloads.tsv')

    stats = {
        'total_rows': 0,
        'invalid_rows': 0,
        'skipped_existing': 0,
        'downloaded': 0,
        'failed': 0,
    }

    with open(tsv_path, 'r', encoding='utf-8') as src, \
         open(failed_log, 'a', encoding='utf-8') as failed_fp, \
         ThreadPoolExecutor(max_workers=int(num_workers)) as executor, \
         tqdm(desc='CC12M rows', unit='row') as pbar:

        futures = {}

        def handle_done(done_set):
            for fut in done_set:
                line_no, image_url, _ = futures.pop(fut)
                try:
                    status, err = fut.result()
                except Exception as exc:
                    status, err = ('failed', f'{type(exc).__name__}: {exc}')

                if status == 'downloaded':
                    stats['downloaded'] += 1
                elif status == 'skipped':
                    stats['skipped_existing'] += 1
                else:
                    stats['failed'] += 1
                    failed_fp.write(f'{line_no}\t{image_url}\t{err}\n')

                pbar.update(1)

        for line_no, raw_line in enumerate(src, start=1):
            stats['total_rows'] += 1
            line = raw_line.rstrip('\n')

            if (not line) or ('	' not in line):
                stats['invalid_rows'] += 1
                pbar.update(1)
                continue

            image_url, caption = line.split('	', 1)
            image_url = image_url.strip()
            caption = caption.strip()

            if (not image_url) or (not caption):
                stats['invalid_rows'] += 1
                pbar.update(1)
                continue

            cache_path = cache_path_from_url(image_url, cache_dir)
            if is_cached(cache_path, verify=verify_existing):
                stats['skipped_existing'] += 1
                pbar.update(1)
                continue

            fut = executor.submit(
                download_to_cache,
                image_url,
                cache_path,
                timeout_seconds,
                retries,
            )
            futures[fut] = (line_no, image_url, cache_path)

            if len(futures) >= int(max_pending):
                done_set, _ = wait(futures.keys(), return_when=FIRST_COMPLETED)
                handle_done(done_set)

        while futures:
            done_set, _ = wait(futures.keys(), return_when=FIRST_COMPLETED)
            handle_done(done_set)

    return stats, failed_log

In [ ]:
stats, failed_log_path = run_cc12m_cache_download(
    tsv_path=TSV_PATH,
    cache_dir=CACHE_DIR,
    num_workers=NUM_WORKERS,
    max_pending=MAX_PENDING,
    timeout_seconds=TIMEOUT_SECONDS,
    retries=RETRIES,
    verify_existing=VERIFY_EXISTING,
)

print('\nDownload summary')
print('total_rows       :', stats['total_rows'])
print('invalid_rows     :', stats['invalid_rows'])
print('skipped_existing :', stats['skipped_existing'])
print('downloaded       :', stats['downloaded'])
print('failed           :', stats['failed'])
if stats['failed'] > 0:
    print('Failed download log:', failed_log_path)

In [1]:
import os
import time
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import Request, urlopen

from tqdm.auto import tqdm


def _default_hf_shard_urls(num_shards=69):
    """Build default Hugging Face WebDataset shard URLs."""
    base_url = (
        "https://huggingface.co/datasets/ProGamerGov/"
        "synthetic-dataset-1m-high-quality-captions/"
        "resolve/main/data/data-{i:06d}.tar"
    )
    return [base_url.format(i=i) for i in range(int(num_shards))]


HF_NUM_SHARDS = 69
HF_DATASET_DIR = Path('dataset') / 'synthetic-dataset-1m-high-quality-captions'
HF_TIMEOUT_SECONDS = 120
HF_RETRIES = 3
HF_CHUNK_SIZE = 1024 * 1024
HF_OVERWRITE = False


def download_hf_shard(
    url,
    output_dir,
    overwrite=False,
    timeout_seconds=120,
    retries=3,
    chunk_size=1024 * 1024,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = Path(urlparse(str(url)).path).name
    output_path = output_dir / filename
    tmp_path = output_path.with_name(output_path.name + '.part')

    if output_path.exists() and output_path.stat().st_size > 0 and not overwrite:
        return 'skipped', output_path

    for attempt in range(1, int(retries) + 1):
        try:
            request = Request(str(url), headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(request, timeout=float(timeout_seconds)) as response:
                total = response.headers.get('Content-Length')
                total = int(total) if total and total.isdigit() else None

                with open(tmp_path, 'wb') as f, tqdm(
                    total=total,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=filename,
                    leave=False,
                ) as pbar:
                    while True:
                        chunk = response.read(int(chunk_size))
                        if not chunk:
                            break
                        f.write(chunk)
                        pbar.update(len(chunk))

            os.replace(tmp_path, output_path)
            return 'downloaded', output_path
        except Exception as exc:
            if attempt >= int(retries):
                raise RuntimeError(f'Failed to download {url}') from exc
            time.sleep(min(2 ** (attempt - 1), 10))


def download_hf_dataset_shards(
    urls=None,
    output_dir=HF_DATASET_DIR,
    num_shards=HF_NUM_SHARDS,
    overwrite=HF_OVERWRITE,
):
    urls = _default_hf_shard_urls(num_shards) if urls is None else list(urls)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary = {'downloaded': 0, 'skipped': 0}
    for url in tqdm(urls, desc='HF shards', unit='shard'):
        status, path = download_hf_shard(
            url,
            output_dir=output_dir,
            overwrite=overwrite,
            timeout_seconds=HF_TIMEOUT_SECONDS,
            retries=HF_RETRIES,
            chunk_size=HF_CHUNK_SIZE,
        )
        summary[status] += 1

    print('Saved shards to:', output_dir.resolve())
    print('downloaded:', summary['downloaded'])
    print('skipped   :', summary['skipped'])
    return summary


hf_summary = download_hf_dataset_shards()

HF shards:   0%|          | 0/69 [00:00<?, ?shard/s]

data-000000.tar:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

data-000001.tar:   0%|          | 0.00/5.67G [00:00<?, ?B/s]

data-000002.tar:   0%|          | 0.00/6.75G [00:00<?, ?B/s]

KeyboardInterrupt: 